# PS3 — Root-Cause Models → S3 → Aurora  (v2)

Same models as v1. What changed is everything downstream of them.

## Why v2 exists

v1 wrote a 797 KB file of `INSERT` statements, which a person copied into the
dashboard-api's deployment package before redeploying the API. That put the
**data inside the code artifact**: every data refresh became a redeploy of the
service that serves the dashboard, there was no S3 object to re-load, diff or
audit afterwards, and none of it could be scheduled.

v2 publishes CSV artifacts to S3 and lets a loader Lambda inside the RDS VPC
put them into Aurora — the same path PS2, PS4 and PS5 already use.

```
notebook ─► s3://…/chicago/ps3/rootcause_outputs/<run_id>/*.csv
         └─► s3://…/chicago/ps3/models/<run_id>/<fleet>/model.joblib
                        │
       cubic-mars-ps3-rc-loader (in the RDS VPC)
                        │
                    Aurora ps3_*  ─►  dashboard
```

**Cell 3 still writes the SQL file.** It is the fallback and it is unchanged —
if the loader is not deployed yet, that path still works exactly as it did.

## What is new

| cell | does |
|---|---|
| 1 | writes `ps3_rc_features.py` — the feature contract shared by training and daily scoring, so the two cannot drift |
| 2 | trains, now importing that module and **keeping the fitted objects** |
| 3 | SQL fallback (unchanged from v1) |
| 4 | publishes the three CSV artifacts + manifest to S3 |
| 5 | packages each fleet's model, encoder and feature order for daily inference |

## The leakage problem, and what we predict instead

`component_subsystem` — the target — is supplied by `silver.dim_event_type`
joined on `EVENT_TYPE_ID`. So `event_type_id`, `event_type_name`,
`event_type_desc` and `component_type_name` are **deterministic functions of the
target**. A model given any of them scores ~100% and has learned a join.

**What we predict instead:** given this device's *failure history* and the day's
context, which component is failing? Every history feature is `shift(1)`-ed
after sorting, so a row can never see its own label. Evaluation is a **time
split**, not random.

## Read lift and kappa, not accuracy

GATE is 86.6% `DOPP` and VALIDATOR 80.7% `COMMS`. On those fleets an accuracy
near the baseline means the model learned to say the common thing. A low lift
here is a finding, not a failure.


In [ ]:
# =============================================================================
# CELL 1 -- write the shared feature contract to disk.
#
# WHY THE NOTEBOOK WRITES ITS OWN MODULE. Training built its features inline in
# v1. If daily scoring rebuilds them from a second copy of that code, the two
# drift -- and training/serving skew does not announce itself. It shows up as a
# model that scored 0.91 here and quietly predicts the majority class in
# production, which is indistinguishable from "the model was always weak".
#
# So both sides import ONE module. This cell materialises it next to the
# notebook; ps3_rc_daily_score.py imports the same file. The artifact written in
# cell 5 records FEATURE_CONTRACT_VERSION, and the scorer refuses to run if it
# does not match -- an abort, not a warning.
# =============================================================================
FEATURES_PY = r'''
"""
ps3_rc_features.py -- the PS3 root-cause feature contract, in ONE place.

WHY THIS FILE EXISTS
--------------------
Training built its features inside the notebook. If daily scoring rebuilds
them from a second copy of that code, the two drift -- and training/serving
skew does not announce itself. It shows up as a model that scored 0.91 in the
notebook and quietly predicts the majority class in production, which is
indistinguishable from "the model was always weak" unless someone happens to
diff two files that were never meant to be compared.

So both sides import this module. The notebook trains with it, the daily
scorer scores with it, and the packaged artifact records the exact feature
ORDER it was fitted on. If this file changes, every artifact fitted before
the change fails its own contract check at load time rather than silently
scoring on shifted columns.

THE LEAKAGE RULE, RESTATED WHERE IT IS ENFORCED
-----------------------------------------------
component_subsystem -- the target -- is supplied by silver.dim_event_type
joined on EVENT_TYPE_ID. So event_type_id, event_type_name, event_type_desc
and component_type_name are deterministic functions of the target. Anything
in LEAKY is barred from the feature list, and assert_no_leakage() is called
on both paths, not just the training one.

Every history feature is shift(1)-ed after sorting on (device_id,
transit_day), so a row can never see its own label. That is the whole reason
the feature builder is backward-looking by construction rather than by
convention.
"""
import numpy as np
import pandas as pd

FEATURE_CONTRACT_VERSION = "ps3_rc_features.2026-08-03.v1"

# Columns that ARE the target, or reconstruct it. Restated here so nobody
# re-adds one later in a notebook cell that never sees the training code.
LEAKY = [
    "component_subsystem", "component_type_name", "event_type_name",
    "event_type_desc", "event_type_id", "root_cause", "root_cause_state",
    "sn_root_cause", "sn_severity", "severity_target", "sn_agrees",
    "sn_fault_desc", "sn_label_state", "component_serial_nbr",
]

# The non-history features, in fixed order. Anything absent from the frame is
# dropped by the caller and RECORDED in the artifact, so a scoring frame that
# is missing a column the model was fitted on is caught rather than filled
# with a plausible zero.
BASE_NUM = [
    "prior_oos_count", "days_since_prior_oos", "device_lifetime_days",
    "prior_oos_rate", "hour_of_day", "day_of_week",
    "events_in_day", "informative_in_day", "component_position",
]
BASE_CAT = ["control_group", "transit_mode", "device_type_name", "prior_root_cause"]

TARGET = "root_cause"
KEYS = ["device_id", "transit_day"]


def prior_share_col(cls):
    return "prior_share_%s" % cls


def build_history_features(d, classes=None):
    """Device failure history as of each device-day, strictly backward-looking.

    `classes` pins WHICH prior_share_* columns are produced and in what order.
    At training time it is None and the classes are read off the data. At
    scoring time it MUST be the trained class list -- otherwise a fleet that
    happened not to see a class in the scoring window produces a narrower
    matrix, every column after the gap shifts by one, and the model scores
    confidently on the wrong numbers. That failure is silent, which is why
    the parameter is not optional in the scorer.
    """
    d = d.sort_values(KEYS).copy()
    g = d.groupby("device_id", sort=False)

    d["prior_oos_count"] = g.cumcount()
    d["days_since_prior_oos"] = (d["transit_day"] - g["transit_day"].shift(1)).dt.days
    d["device_lifetime_days"] = (
        d["transit_day"] - g["transit_day"].transform("min")).dt.days
    d["prior_oos_rate"] = d["prior_oos_count"] / d["device_lifetime_days"].replace(0, np.nan)

    # the device's PREVIOUS root cause ...
    d["prior_root_cause"] = g[TARGET].shift(1)

    # ... and how often each cause has occurred for it BEFORE this row.
    # shift(1) then expanding().mean() is the whole guarantee: the row's own
    # label is excluded from its own feature.
    if classes is None:
        classes = sorted(d[TARGET].dropna().unique())
    for cls in classes:
        hit = (d[TARGET] == cls).astype(float)
        d[prior_share_col(cls)] = (
            hit.groupby(d["device_id"])
               .apply(lambda s: s.shift(1).expanding().mean())
               .reset_index(level=0, drop=True))
    return d


def feature_lists(d, classes):
    """The (numeric, categorical) feature names, in the order the model sees.

    Order is part of the contract. numpy has no column names, so a reordered
    list is not an error -- it is a wrong answer.
    """
    feat_num = [c for c in BASE_NUM if c in d.columns]
    feat_num += [prior_share_col(c) for c in classes if prior_share_col(c) in d.columns]
    feat_cat = [c for c in BASE_CAT if c in d.columns]
    assert_no_leakage(feat_num + feat_cat)
    return feat_num, feat_cat


def assert_no_leakage(features):
    leaked = [c for c in features if c in LEAKY]
    if leaked:
        raise AssertionError(
            "LEAKAGE: %s would reconstruct the target. component_subsystem comes "
            "from dim_event_type joined on EVENT_TYPE_ID, so any event-type column "
            "IS the label wearing a different name." % leaked)


def design_matrix(d, feat_num, feat_cat, encoder, fit=False):
    """Assemble X in the contracted order, or say exactly what is missing.

    A missing column is NOT filled with a zero here. A zero in
    prior_oos_count means "this device has never failed before", which is a
    real and confident statement; inventing it because a column did not
    arrive turns a pipeline fault into a prediction.
    """
    missing = [c for c in feat_num + feat_cat if c not in d.columns]
    if missing:
        raise KeyError(
            "scoring frame is missing %d contracted feature(s): %s. The model was "
            "fitted on them; scoring without them would silently shift every "
            "column after the gap." % (len(missing), missing))

    num = d[feat_num].to_numpy(dtype=float)
    if not feat_cat:
        return num
    cat_raw = d[feat_cat].astype(str)
    cat = encoder.fit_transform(cat_raw) if fit else encoder.transform(cat_raw)
    return np.hstack([num, cat])


def prepare_for_scoring(spine, score_dates, classes):
    """History from labelled days, rows to score from the requested days.

    THE SUBTLETY THAT MATTERS. Training filtered to root_cause_state ==
    'LABELLED' and built history over exactly those rows. Scoring must build
    history the same way -- but the days being scored are, by definition, the
    ones whose cause is not yet known, so they cannot be inside that filter.

    So: take every LABELLED day strictly before the scoring window as
    history, append the scoring days regardless of their label state, sort,
    build features over the union, then return only the scoring rows. Because
    every feature is shift(1)-ed, a scoring row reads its device's past and
    never itself.
    """
    spine = spine.copy()
    spine["transit_day"] = pd.to_datetime(spine["transit_day"])
    want = pd.to_datetime(pd.Series(list(score_dates))).dt.normalize()
    lo = want.min()

    hist = spine[(spine["root_cause_state"] == "LABELLED") & (spine["transit_day"] < lo)]
    todo = spine[spine["transit_day"].dt.normalize().isin(set(want))]
    if todo.empty:
        return todo, todo

    both = pd.concat([hist, todo], ignore_index=True)
    feats = build_history_features(both, classes=classes)
    mask = feats["transit_day"].dt.normalize().isin(set(want))
    return feats[mask].copy(), feats

'''

with open("ps3_rc_features.py", "w") as fh:
    fh.write(FEATURES_PY.lstrip("\n"))

import importlib, ps3_rc_features as F
importlib.reload(F)
print("wrote ps3_rc_features.py")
print("  contract version :", F.FEATURE_CONTRACT_VERSION)
print("  base numeric     :", len(F.BASE_NUM), "features")
print("  base categorical :", len(F.BASE_CAT), "features")
print('  barred as leaky  :', len(F.LEAKY), 'columns')


In [ ]:
# =============================================================================
# CELL 2 -- ROOT-CAUSE MODELS, all three fleets, LEAKAGE-SAFE
#
# Identical modelling to v1. Two changes, both in service of cell 5:
#   1. features come from ps3_rc_features, so training and scoring share one
#      implementation instead of two copies that look the same today;
#   2. run_fleet() KEEPS the fitted classifier, the encoder and the exact
#      feature ORDER. numpy has no column names -- a reordered feature list is
#      not an error, it is a wrong answer -- so the order is part of what gets
#      packaged.
#
# THE HONEST QUESTION: given this device's FAILURE HISTORY and the context of
# the day, which component is failing? "This device has failed on the card
# reader three times in the last 90 days" is real evidence, available before the
# event code is read, and it is what lets PS3 chain onto PS1's three-day-ahead
# prediction.
#
# EVALUATION IS TIME-SPLIT, not random. Device history leaks across a random
# split -- the same device appears either side of it.
# =============================================================================
import pandas as pd, numpy as np, json, os, time
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             cohen_kappa_score, matthews_corrcoef,
                             classification_report)
import ps3_rc_features as F

OUT = "PS3_spine_outputs"
SUB = {"TVM": "tvm", "GATE": "gates", "VALIDATOR": "validators"}

TEST_FRACTION  = 0.25     # last 25% of the window, by date
MIN_CLASS_ROWS = 50       # a class the model can never learn is noise


def run_fleet(fleet):
    print("=" * 84)
    print(f"PS3 ROOT-CAUSE MODEL -- {fleet}")
    print("=" * 84)
    d = pd.read_parquet(f"{OUT}/{SUB[fleet]}_ps3_spine.parquet")
    d["transit_day"] = pd.to_datetime(d["transit_day"])
    d = d[d["root_cause_state"] == "LABELLED"].copy()
    print(f"  labelled device-days: {len(d):,}")

    keep = d["root_cause"].value_counts()
    keep = keep[keep >= MIN_CLASS_ROWS].index
    dropped = d[~d["root_cause"].isin(keep)]
    if len(dropped):
        print(f"  dropping {len(dropped):,} rows in classes under {MIN_CLASS_ROWS}: "
              f"{sorted(set(dropped['root_cause']))}")
    d = d[d["root_cause"].isin(keep)].copy()

    # The class list is PINNED here and travels with the model. At scoring time
    # the prior_share_* columns are rebuilt from this list, not from whatever
    # the scoring window happens to contain -- otherwise a fleet that saw no
    # CARD_READER failures that week produces a narrower matrix, every column
    # after the gap shifts by one, and the model scores confidently on the
    # wrong numbers. Silently.
    classes = sorted(d["root_cause"].dropna().unique())
    d = F.build_history_features(d, classes=classes)
    feat_num, feat_cat = F.feature_lists(d, classes)

    cut = d["transit_day"].quantile(1 - TEST_FRACTION)
    tr, te = d[d["transit_day"] <= cut], d[d["transit_day"] > cut]
    print(f"  time split at {cut.date()}  train {len(tr):,} | test {len(te):,}")
    if len(te) < 100 or tr["root_cause"].nunique() < 2:
        print("  !! insufficient test data or single class -- skipping")
        return None

    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    Xtr = F.design_matrix(tr, feat_num, feat_cat, enc, fit=True)
    Xte = F.design_matrix(te, feat_num, feat_cat, enc, fit=False)
    ytr, yte = tr["root_cause"].to_numpy(), te["root_cause"].to_numpy()

    t0 = time.time()
    clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1,
                                         max_depth=6, random_state=42,
                                         categorical_features=None)
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    secs = time.time() - t0

    base_cls = pd.Series(ytr).value_counts().index[0]
    base_acc = (yte == base_cls).mean()
    acc  = accuracy_score(yte, pred)
    f1m  = f1_score(yte, pred, average="macro", zero_division=0)
    f1w  = f1_score(yte, pred, average="weighted", zero_division=0)
    bal  = balanced_accuracy_score(yte, pred)
    kap  = cohen_kappa_score(yte, pred)
    mcc  = matthews_corrcoef(yte, pred)

    print(f"\n  trained in {secs:.0f}s | {len(feat_num)+len(feat_cat)} features, "
          f"{len(set(ytr))} classes")
    print(f"  {'accuracy':22s} {acc:.4f}")
    print(f"  {'majority baseline':22s} {base_acc:.4f}   (always predict {base_cls})")
    print(f"  {'LIFT over baseline':22s} {acc-base_acc:+.4f}   <- the number that matters")
    print(f"  {'macro F1':22s} {f1m:.4f}")
    print(f"  {'weighted F1':22s} {f1w:.4f}")
    print(f"  {'balanced accuracy':22s} {bal:.4f}")
    print(f"  {'Cohen kappa':22s} {kap:.4f}   (0 = no better than chance)")
    print(f"  {'MCC':22s} {mcc:.4f}")
    if acc - base_acc < 0.02:
        print("\n  !! NO MEANINGFUL LIFT. Device history does not predict which")
        print("     component fails. The diagnostic information lives in the event")
        print("     code, not in the device's past -- which is itself the finding.")
    print("\n  per class:")
    print(classification_report(yte, pred, zero_division=0))

    return {"fleet": fleet, "n_train": int(len(tr)), "n_test": int(len(te)),
            "n_features": int(len(feat_num) + len(feat_cat)),
            "n_classes": int(len(set(ytr))),
            "class_labels": ",".join(sorted(set(ytr))),
            "accuracy": round(float(acc), 4), "baseline": round(float(base_acc), 4),
            "lift": round(float(acc - base_acc), 4),
            "f1_macro": round(float(f1m), 4), "f1_weighted": round(float(f1w), 4),
            "balanced_accuracy": round(float(bal), 4),
            "cohen_kappa": round(float(kap), 4), "mcc": round(float(mcc), 4),
            "champion": "HistGradientBoosting", "split_date": str(cut.date()),
            "predictions": te.assign(pred_root_cause=pred),
            # --- what cell 5 packages -------------------------------------
            "clf": clf, "encoder": enc, "classes": list(classes),
            "feat_num": list(feat_num), "feat_cat": list(feat_cat)}


results = {}
for fleet in SUB:
    r = run_fleet(fleet)
    if r: results[fleet] = r
    print()

print("=" * 84)
print("PS3 ROOT-CAUSE MODEL SUMMARY -- all fleets")
print("=" * 84)
print(f"  {'fleet':10s} {'classes':>8s} {'accuracy':>9s} {'baseline':>9s} {'lift':>8s} "
      f"{'macroF1':>8s} {'kappa':>7s}")
for f, r in results.items():
    print(f"  {f:10s} {r['n_classes']:8d} {r['accuracy']:9.4f} {r['baseline']:9.4f} "
          f"{r['lift']:+8.4f} {r['f1_macro']:8.4f} {r['cohen_kappa']:7.4f}")
print("""
  READ LIFT AND KAPPA, NOT ACCURACY. On GATE and VALIDATOR one class is ~80-87%
  of the data, so accuracy near the baseline means the model has learned to say
  the common thing. Cohen's kappa near 0 says the same in one number.""")


In [ ]:
# =============================================================================
# CELL 3 -- SQL FALLBACK (unchanged from v1)
#
# Kept because it works and it is the path the demo ran on. Cells 4 and 5
# are the one that gets scheduled. If the loader Lambda is deployed, this
# file is a belt-and-braces artifact; if it is not, this is still the way in.
#
# PS3 -- EXPORT TO AURORA  (dashboard load file)
#
# The dashboard API loads PS3 from SQL files under sql/load/, not parquet:
#     aws lambda invoke ... --payload '{"action":"load_run","only":["ps3"]}'
# So this writes INSERT statements matching the live Aurora DDL exactly.
#
# Tables written:
#   ps3_model_runs           one row  -- run provenance
#   ps3_head_summary         one row per fleet -- metrics the dashboard renders
#   ps3_device_predictions   one row per device -- the drill-down
#
# Column names and types are taken from the deployed DDL in
# sql/15_phase2a_ps3_two_head.sql. If a load fails on a column, that DDL is the
# source of truth, not this file.
# =============================================================================
import pandas as pd, numpy as np, datetime as dt, os

CITY        = "CHI"
AS_OF       = "2026-04-11"                     # matches the spine RUN_DATE
RUN_ID      = f"ps3_oos_{dt.datetime.now().strftime('%Y%m%d')}"
NOTEBOOK    = "PS3_03_RootCause_Models.ipynb"
MACRO_F1_FLOOR = 0.45                          # the existing PS3 root_cause gate

def q(v):
    """SQL literal. None/NaN -> NULL, strings escaped."""
    if v is None or (isinstance(v, float) and np.isnan(v)): return "NULL"
    if isinstance(v, (bool, np.bool_)):  return "TRUE" if v else "FALSE"
    if isinstance(v, (int, np.integer)): return str(int(v))
    if isinstance(v, (float, np.floating)): return f"{float(v):.6f}"
    return "'" + str(v).replace("'", "''") + "'"

lines = [
    "-- =========================================================================",
    f"-- PS3 root-cause run {RUN_ID}",
    f"-- generated {dt.datetime.now().isoformat(timespec='seconds')}",
    "--",
    "-- Event contract: docs/OOS_EVENT_CONTRACT.md (2026-08-03.v5)",
    "--   silver.device_event_enriched | is_hardware_oos_event=TRUE",
    "--   | UPPER(TRIM(EVENT_STATE_TYPE_NAME))='SET' | calendar-day grain",
    "--",
    "-- Supersedes ps3_20260726, which sat on silver.incident_root_cause",
    "-- (ServiceNow availability events, the CHARGEABLE subset) and therefore",
    "-- had 0 validator rows by construction.",
    "-- =========================================================================",
    "BEGIN;",
    "",
    f"DELETE FROM ps3_device_predictions WHERE city_id = {q(CITY)} AND run_id = {q(RUN_ID)};",
    f"DELETE FROM ps3_head_summary       WHERE city_id = {q(CITY)} AND run_id = {q(RUN_ID)};",
    f"DELETE FROM ps3_model_runs         WHERE city_id = {q(CITY)} AND run_id = {q(RUN_ID)};",
    "",
]

# --- ps3_model_runs ----------------------------------------------------------
n_inc = int(sum(r["n_train"] + r["n_test"] for r in results.values()))
lines += [
    "-- run provenance -----------------------------------------------------------",
    "INSERT INTO ps3_model_runs (city_id, run_id, run_ts, run_kind, source_notebook,",
    "  n_incidents, device_scope, mlflow_version, severity_collapse_verified, notes, as_of_date)",
    "VALUES (" + ", ".join([
        q(CITY), q(RUN_ID), q(dt.datetime.now().isoformat(timespec='seconds')),
        q("train"), q(NOTEBOOK), q(n_inc), q(",".join(results.keys())), q("v3"),
        q(False),
        q("Rebuilt on the hardware-OOS contract. Root cause = device-native "
          "component_subsystem. Event-type columns EXCLUDED as leakage. "
          "Time-split evaluation."),
        q(AS_OF)]) + ")",
    "ON CONFLICT (city_id, run_id) DO NOTHING;",
    "",
]

# --- ps3_head_summary --------------------------------------------------------
lines += ["-- per-fleet metrics --------------------------------------------------------"]
for fleet, r in results.items():
    gate_pass = bool(r["f1_macro"] >= MACRO_F1_FLOOR)
    lines += [
        "INSERT INTO ps3_head_summary (city_id, run_id, device_category, head, modeled,",
        "  target_col, champion, n_classes, class_labels, test_f1_macro, test_f1_weighted,",
        "  test_accuracy, test_balanced_accuracy, test_cohen_kappa, test_mcc,",
        "  macro_f1_floor, gate_pass, n_train, n_test, n_features, rootcause_source, as_of_date)",
        "VALUES (" + ", ".join([
            q(CITY), q(RUN_ID), q(fleet), q("root_cause"), q(True),
            q("component_subsystem"), q(r["champion"]), q(r["n_classes"]),
            q(r["class_labels"]), q(r["f1_macro"]), q(r["f1_weighted"]),
            q(r["accuracy"]), q(r["balanced_accuracy"]), q(r["cohen_kappa"]),
            q(r["mcc"]), q(MACRO_F1_FLOOR), q(gate_pass),
            q(r["n_train"]), q(r["n_test"]), q(r["n_features"]),
            q("device_event_enriched"), q(AS_OF)]) + ")",
        "ON CONFLICT (city_id, run_id, device_category, head) DO NOTHING;",
        "",
    ]

# --- ps3_device_predictions --------------------------------------------------
lines += ["-- per-device drill-down ----------------------------------------------------"]
rowct = 0
for fleet, r in results.items():
    p = r["predictions"]
    agg = (p.groupby("device_id")
             .agg(n_incidents=("device_id", "size"),
                  dominant_pred_component=("pred_root_cause",
                                           lambda s: s.value_counts().index[0]),
                  last_incident_dtm=("event_dtm", "max"))
             .reset_index())
    for _, row in agg.iterrows():
        lines.append(
            "INSERT INTO ps3_device_predictions (city_id, run_id, device_id, "
            "mars_device_category, n_incidents, dominant_pred_component, "
            "last_incident_dtm, computed_date) VALUES ("
            + ", ".join([q(CITY), q(RUN_ID), q(row["device_id"]), q(fleet),
                         q(int(row["n_incidents"])), q(row["dominant_pred_component"]),
                         q(str(row["last_incident_dtm"])), q(AS_OF)])
            + ") ON CONFLICT (city_id, run_id, device_id) DO NOTHING;")
        rowct += 1

lines += ["", "COMMIT;", ""]

os.makedirs("PS3_sql_outputs", exist_ok=True)
path = f"PS3_sql_outputs/ps3_run_{dt.datetime.now().strftime('%Y%m%d')}.sql"
with open(path, "w") as fh:
    fh.write("\n".join(lines))

print("=" * 84)
print("SQL LOAD FILE WRITTEN")
print("=" * 84)
print(f"  {path}")
print(f"  {os.path.getsize(path):,} bytes")
print(f"  ps3_model_runs         1 row")
print(f"  ps3_head_summary       {len(results)} rows")
print(f"  ps3_device_predictions {rowct:,} rows")
print(f"\n  run_id: {RUN_ID}")
print(f"""
  TO LOAD IT
    1. copy this file into the repo at
         api/lambda/cubic-mars-dashboard-api/sql/load/
    2. add it to the LOAD_FILES list in handler.py next to the ps3 entry
    3. redeploy the dashboard-api and invoke:
         aws lambda invoke --function-name cubic-mars-dashboard-api \\
           --cli-binary-format raw-in-base64-out \\
           --payload '{{"action":"load_run","only":["ps3"]}}' out.json

  The DELETEs at the top make it idempotent -- re-running replaces this run_id
  rather than duplicating it.""")
print("=" * 84)


In [ ]:
# =============================================================================
# CELL 4 -- PUBLISH THE RUN TO S3   (this is the path that gets scheduled)
#
# Writes the three tables cell 3 writes as INSERTs, plus ps3_serial_predictions
# (04-Aug-2026), as CSV artifacts
# under one run_id, plus a manifest. cubic-mars-ps3-rc-loader then reads them
# from inside the RDS VPC and loads Aurora transactionally.
#
# WHY A MANIFEST. The loader refuses to load a run that is missing an artifact.
# A half-loaded PS3 puts today's metrics beside last week's device list and
# NOTHING on the dashboard reveals that -- whereas a stale run is visible on
# /ps3/status. The manifest is what makes "complete" checkable rather than
# assumed.
#
# WHY CSV AND NOT PARQUET. Three small tables, and CSV means the loader needs no
# pyarrow layer. The PS3 v25 feed publishes parquet because its widest table is
# 54,239 x 79 and INSERT could not finish it; this head is 2,500 rows.
# =============================================================================
import boto3, io, json, datetime as dt, pandas as pd

ARTIFACT_BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
RC_PREFIX       = "chicago/ps3/rootcause_outputs"
CITY            = "CHI"
AS_OF           = "2026-04-11"                 # matches the spine RUN_DATE
RUN_ID          = f"ps3_oos_{dt.datetime.now().strftime('%Y%m%d')}"
NOTEBOOK        = "PS3_03_RootCause_Models_v2.ipynb"
MACRO_F1_FLOOR  = 0.45
NOW             = dt.datetime.now().isoformat(timespec="seconds")

s3 = boto3.client("s3")

def put(key, body, ctype="text/csv"):
    s3.put_object(Bucket=ARTIFACT_BUCKET, Key=key, Body=body, ContentType=ctype)
    return f"s3://{ARTIFACT_BUCKET}/{key}"

def csv_bytes(df):
    b = io.StringIO(); df.to_csv(b, index=False); return b.getvalue().encode("utf-8")

# ---- ps3_model_runs ---------------------------------------------------------
runs_df = pd.DataFrame([{
    "city_id": CITY, "run_id": RUN_ID, "run_ts": NOW, "run_kind": "train",
    "source_notebook": NOTEBOOK,
    "n_incidents": int(sum(r["n_train"] + r["n_test"] for r in results.values())),
    "device_scope": ",".join(results.keys()),
    "mlflow_version": "v3", "severity_collapse_verified": False,
    "notes": ("Rebuilt on the hardware-OOS contract. Root cause = device-native "
              "component_subsystem. Event-type columns EXCLUDED as leakage. "
              f"Time-split evaluation. Feature contract {F.FEATURE_CONTRACT_VERSION}."),
    "as_of_date": AS_OF,
}])

# ---- ps3_head_summary -------------------------------------------------------
head_df = pd.DataFrame([{
    "city_id": CITY, "run_id": RUN_ID, "device_category": fleet,
    "head": "root_cause", "modeled": True,
    "target_col": "component_subsystem", "champion": r["champion"],
    "n_classes": r["n_classes"], "class_labels": r["class_labels"],
    "test_f1_macro": r["f1_macro"], "test_f1_weighted": r["f1_weighted"],
    "test_accuracy": r["accuracy"], "test_balanced_accuracy": r["balanced_accuracy"],
    "test_cohen_kappa": r["cohen_kappa"], "test_mcc": r["mcc"],
    "macro_f1_floor": MACRO_F1_FLOOR,
    "gate_pass": bool(r["f1_macro"] >= MACRO_F1_FLOOR),
    "n_train": r["n_train"], "n_test": r["n_test"], "n_features": r["n_features"],
    "rootcause_source": "device_event_enriched",
    "as_of_date": AS_OF,
} for fleet, r in results.items()])

# ---- ps3_device_predictions -------------------------------------------------
dev = []
for fleet, r in results.items():
    agg = (r["predictions"].groupby("device_id")
             .agg(n_incidents=("device_id", "size"),
                  dominant_pred_component=("pred_root_cause",
                                           lambda s: s.value_counts().index[0]),
                  last_incident_dtm=("event_dtm", "max"))
             .reset_index())
    agg["city_id"] = CITY; agg["run_id"] = RUN_ID
    agg["mars_device_category"] = fleet; agg["computed_date"] = AS_OF
    dev.append(agg)
dev_df = pd.concat(dev, ignore_index=True)[
    ["city_id", "run_id", "device_id", "mars_device_category", "n_incidents",
     "dominant_pred_component", "last_incident_dtm", "computed_date"]]

# ---- ps3_serial_predictions -------------------------------------------------
# SERIAL GRAIN, added 04-Aug-2026, so Device 360's component panel has rows to
# join to. Until this artifact existed that panel stamped every component
# "not_in_feed", which a reader takes for a lookup failure rather than for an
# absent table.
#
# THIS IS NOT THE DEVICE PREDICTION BROADCAST ONTO ITS SERIALS. Every episode
# in r["predictions"] already carries its OWN predicted component and the
# serial it was raised against, so the mode below is taken within each
# serial's own episodes. Copying the device's dominant component onto all of
# its serials would produce a table that looks per-component and is not --
# more precise on screen than in fact, which is the exact failure this run was
# rebuilt to avoid.
#
# USING component_serial_nbr HERE DOES NOT CONTRADICT ITS PLACE IN THE FEATURE
# BUILDER'S LEAKY LIST. It is barred as a FEATURE because it reconstructs the
# target. Using it as a GROUPING KEY for reporting is a different act: no
# model sees it, and assert_no_leakage() is unaffected.
#
# ROWS WITH NO SERIAL ARE DROPPED, NOT DEFAULTED. matched_serial_nbr is
# NOT NULL and part of the primary key (city_id, run_id, device_id,
# matched_serial_nbr). A placeholder would either collide on that key or
# invent a component that does not exist. The dropped count is printed and
# carried in the manifest, so coverage is visible rather than implied.
SER_COL = "component_serial_nbr"
_ser, _ser_dropped, _ser_total = [], 0, 0
for fleet, r in results.items():
    pr = r["predictions"]
    _ser_total += len(pr)
    if SER_COL not in pr.columns:
        _ser_dropped += len(pr)
        print(f"  !! {fleet}: no {SER_COL} column on the prediction frame")
        continue
    keyed = pr[pr[SER_COL].notna() & (pr[SER_COL].astype(str).str.strip() != "")]
    _ser_dropped += len(pr) - len(keyed)
    if keyed.empty:
        continue
    agg = (keyed.groupby(["device_id", SER_COL])
             .agg(n_incidents=("device_id", "size"),
                  dominant_pred_component=("pred_root_cause",
                                           lambda s: s.value_counts().index[0]),
                  last_incident_dtm=("event_dtm", "max"))
             .reset_index()
             .rename(columns={SER_COL: "matched_serial_nbr"}))
    agg["city_id"] = CITY; agg["run_id"] = RUN_ID
    agg["mars_device_category"] = fleet; agg["computed_date"] = AS_OF
    # Left NULL rather than guessed: the severity head is not modelled in this
    # run, and the episode frame carries no component age.
    agg["pct_critical_pred"] = None
    agg["component_age_days"] = None
    _ser.append(agg)

SER_COLS = ["city_id", "run_id", "device_id", "matched_serial_nbr",
            "mars_device_category", "n_incidents", "component_age_days",
            "dominant_pred_component", "pct_critical_pred",
            "last_incident_dtm", "computed_date"]
ser_df = (pd.concat(_ser, ignore_index=True) if _ser
          else pd.DataFrame(columns=SER_COLS))
ser_df = ser_df.reindex(columns=SER_COLS)

# WIDTH GUARD, and it is here because of a real failure. A 41-character value
# in a VARCHAR(40) raised 22001 mid-load on 04-Aug and the loader rolled back
# the ENTIRE run -- correct behaviour, and a dry run could not have caught it
# because a dry run never INSERTs. Over-length values are dropped with a loud
# count rather than truncated: a truncated serial is a serial that matches the
# wrong component, which is worse than a missing row.
for _col, _max in (("matched_serial_nbr", 64), ("device_id", 40),
                   ("dominant_pred_component", 48), ("mars_device_category", 12)):
    if _col in ser_df.columns and len(ser_df):
        _over = ser_df[_col].astype(str).str.len() > _max
        if _over.any():
            print(f"  !! {int(_over.sum())} row(s) with {_col} longer than "
                  f"{_max} chars -- DROPPED, not truncated "
                  f"(max seen {int(ser_df[_col].astype(str).str.len().max())})")
            ser_df = ser_df[~_over]

_ser_devices = int(ser_df["device_id"].nunique()) if len(ser_df) else 0
print(f"  serial grain: {len(ser_df):,} rows over {_ser_devices:,} devices; "
      f"{_ser_dropped:,} of {_ser_total:,} episodes had no usable serial "
      f"({(_ser_dropped / max(_ser_total, 1)):.1%})")

base = f"{RC_PREFIX}/{RUN_ID}"
written = [put(f"{base}/ps3_model_runs.csv", csv_bytes(runs_df)),
           put(f"{base}/ps3_head_summary.csv", csv_bytes(head_df)),
           put(f"{base}/ps3_device_predictions.csv", csv_bytes(dev_df)),
           put(f"{base}/ps3_serial_predictions.csv", csv_bytes(ser_df))]

manifest = {"run_id": RUN_ID, "run_kind": "train", "city_id": CITY,
            "as_of_date": AS_OF, "generated": NOW, "source_notebook": NOTEBOOK,
            "feature_contract_version": F.FEATURE_CONTRACT_VERSION,
            "event_contract": "docs/OOS_EVENT_CONTRACT.md 2026-08-03.v5",
            "artifacts": {"ps3_model_runs": len(runs_df),
                          "ps3_head_summary": len(head_df),
                          "ps3_device_predictions": len(dev_df),
                          "ps3_serial_predictions": len(ser_df)},
            "serial_coverage": {"rows": int(len(ser_df)),
                                "devices": _ser_devices,
                                "episodes_without_serial": int(_ser_dropped),
                                "episodes_total": int(_ser_total)},
            "fleets": list(results.keys())}
written.append(put(f"{base}/manifest.json",
                   json.dumps(manifest, indent=2).encode("utf-8"), "application/json"))

print("=" * 84)
print("PUBLISHED TO S3")
print("=" * 84)
for w in written: print(" ", w)
print(f"""
  run_id: {RUN_ID}

  LOAD IT
    AWS_MAX_ATTEMPTS=1 aws lambda invoke --function-name cubic-mars-ps3-rc-loader \\
      --region us-east-1 --cli-binary-format raw-in-base64-out \\
      --payload '{{"run_id":"{RUN_ID}"}}' /tmp/ps3rc.json
    python3 -m json.tool /tmp/ps3rc.json

  DRY RUN FIRST -- reads, shapes, checks every guard, writes nothing, and
  reports S3 row counts against what is already in Aurora:
    --payload '{{"run_id":"{RUN_ID}","dry_run":true}}'""")
print("=" * 84)


In [ ]:
# =============================================================================
# CELL 5 -- PACKAGE THE MODELS FOR DAILY INFERENCE
#
# A fitted estimator on its own is not a servable model. What has to survive
# the notebook is the whole decision procedure:
#
#   clf              the classifier
#   encoder          the FITTED OrdinalEncoder -- refitting it at scoring time
#                    remaps every category and is the classic way to ship a
#                    model that scores well in the notebook and is nonsense in
#                    production
#   feat_num/feat_cat the exact column ORDER. numpy has no column names.
#   classes          pins the prior_share_* columns, so a quiet scoring window
#                    cannot narrow the matrix and shift every column after it
#   contract version so the scorer can REFUSE to run against a feature builder
#                    that has changed underneath it
#   metrics          carried forward, because a scoring day has no labels and
#                    measures nothing -- the dashboard's quality numbers have to
#                    come from the training run and say so
#
# Then: python ps3_rc_daily_score.py --model-run <RUN_ID> --score-date latest
#                                    --invoke-loader
# =============================================================================
import joblib, io, json, boto3, sklearn, sys

MODEL_PREFIX = "chicago/ps3/models"
s3 = boto3.client("s3")

manifest = {"run_id": RUN_ID, "generated": NOW, "city_id": CITY,
            "feature_contract_version": F.FEATURE_CONTRACT_VERSION,
            "sklearn_version": sklearn.__version__,
            "python_version": sys.version.split()[0], "fleets": {}}

print("=" * 84)
print("PACKAGING MODELS")
print("=" * 84)
for fleet, r in results.items():
    base = f"{MODEL_PREFIX}/{RUN_ID}/{fleet}"

    buf = io.BytesIO()
    joblib.dump({"clf": r["clf"], "encoder": r["encoder"]}, buf)
    buf.seek(0)
    s3.put_object(Bucket=ARTIFACT_BUCKET, Key=f"{base}/model.joblib", Body=buf.getvalue())

    meta = {
        "fleet": fleet, "run_id": RUN_ID, "champion": r["champion"],
        "feature_contract_version": F.FEATURE_CONTRACT_VERSION,
        "sklearn_version": sklearn.__version__,
        "classes": r["classes"], "feat_num": r["feat_num"], "feat_cat": r["feat_cat"],
        "n_train": r["n_train"], "n_test": r["n_test"],
        "split_date": r["split_date"], "macro_f1_floor": MACRO_F1_FLOOR,
        "metrics": {k: r[k] for k in ("accuracy", "baseline", "lift", "f1_macro",
                                      "f1_weighted", "balanced_accuracy",
                                      "cohen_kappa", "mcc")},
        "target": "component_subsystem (device-native, via root_cause)",
        "leakage_barred": F.LEAKY,
        "note": ("Metrics are TRAINING metrics measured on the held-out time "
                 "split. A batch scoring run re-uses them and marks itself "
                 "run_kind='batch_score'; it does not re-measure anything, "
                 "because the day it scores has no labels yet."),
    }
    s3.put_object(Bucket=ARTIFACT_BUCKET, Key=f"{base}/metadata.json",
                  Body=json.dumps(meta, indent=2).encode("utf-8"),
                  ContentType="application/json")

    manifest["fleets"][fleet] = {
        "classes": len(r["classes"]),
        "features": len(r["feat_num"]) + len(r["feat_cat"]),
        "f1_macro": r["f1_macro"], "lift": r["lift"],
        "gate_pass": bool(r["f1_macro"] >= MACRO_F1_FLOOR)}
    print(f"  {fleet:10s} -> s3://{ARTIFACT_BUCKET}/{base}/"
          f"   {len(r['feat_num'])+len(r['feat_cat'])} features, "
          f"{len(r['classes'])} classes")

s3.put_object(Bucket=ARTIFACT_BUCKET, Key=f"{MODEL_PREFIX}/{RUN_ID}/manifest.json",
              Body=json.dumps(manifest, indent=2).encode("utf-8"),
              ContentType="application/json")

print(f"""
  manifest: s3://{ARTIFACT_BUCKET}/{MODEL_PREFIX}/{RUN_ID}/manifest.json

  SCORE DAILY
    python ps3_rc_daily_score.py \\
      --model-run {RUN_ID} --score-date latest --invoke-loader

  The scorer imports ps3_rc_features.py -- the same file cell 1 wrote -- and
  ABORTS if its FEATURE_CONTRACT_VERSION does not match what is recorded above.
  A model scored on shifted columns produces confident, plausible, wrong
  answers, and nothing downstream can tell that from a weak model.""")
print("=" * 84)


## Deploying the loader (once)

```bash
# in AWS CloudShell, us-east-1
cd ~ && rm -rf ps3rc && mkdir ps3rc && cd ps3rc
#   upload handler.py from api/lambda/cubic-mars-ps3-rc-loader/
bash deploy.sh
```

Then every run is:

```bash
python ps3_rc_daily_score.py --model-run latest --score-date latest --invoke-loader
```

## Scheduling it

The scorer needs the spine, so it belongs wherever the spine is built — a
SageMaker Processing job or a Glue Python-shell job on a daily EventBridge
rule. The **loader** is the only piece that has to sit inside the RDS VPC, and
it is already there.

Two triggers are worth having, and they are not the same trigger:

1. **After the spine rebuilds** — score, publish, invoke the loader. This is
   the real daily path.
2. **S3 `ObjectCreated` on `rootcause_outputs/*/manifest.json`** — loads
   whatever was just published, including a re-publish or a manual backfill,
   without anyone remembering to invoke anything.

## What this run supersedes

`ps3_20260726` sat on `silver.incident_root_cause` — ServiceNow availability
events, the chargeable subset — and had **zero validator rows by
construction**. This run is on the hardware-OOS contract and covers all three
fleets.
